# A6：Constraint-First 架構 — VPC 限制下的 GCP AI 系統

> **對應 Blog：** FDE 面試準備指南（三十）Constraint-First 架構設計：VPC 限制下的 GCP AI 系統  
> **核心目標：** 學習如何從「限制」出發設計架構，而不是從「最優解」出發。本 Notebook 以 Python 模擬各個 GCP 組件，附上真實的 GCP CLI 指令供實際部署參考。

## 面試情境

> 「你的客戶是一家銀行。IT 安全政策規定：所有含有客戶 PII 的資料不能傳送到外部網路，所有 API 呼叫必須在私有網路內完成，並且每個 AI 模型的調用都要有審計日誌。他們想在這個條件下部署一個 RAG-based 合約審閱 Agent。你的架構是什麼？」

## 學習路徑

| Part | 內容 | 模擬 vs GCP 實際 |
|------|------|------------------|
| 1 | 限制分類框架 | 純概念 |
| 2 | PII 偵測與遮罩 | Python 模擬 + Cloud DLP 指令 |
| 3 | Audit Log 設計 | Python 模擬 + GCP Audit Logs 設定 |
| 4 | VPC-SC 架構說明 | GCP CLI 指令參考 |
| 5 | Private Service Connect | GCP CLI 指令參考 |
| 6 | 完整 RAG Pipeline（VPC 限制版） | Python 模擬 |
| 7 | 面試白板語言 | |

In [ ]:
import re
import json
import time
import uuid
import hashlib
import asyncio
from typing import Any
from dataclasses import dataclass, field
from datetime import datetime, timezone

# 如果有 OpenAI API 可以啟用以下
try:
    from dotenv import load_dotenv
    load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，部分 Cell 將使用模擬輸出")

print("環境就緒")

## Part 1：限制分類框架

FDE 面對限制的第一步：**分類**，不是立刻開始設計架構。

In [ ]:
# 限制分類框架

BANK_CONSTRAINTS = {
    "網路隔離（Network Isolation）": {
        "constraint": "所有 API 呼叫必須在私有網路內完成",
        "gcp_solution": "VPC Service Controls + Private Service Connect",
        "impact": "Vertex AI endpoint 只能透過私有 IP 存取"
    },
    "資料分類（Data Classification）": {
        "constraint": "PII 不能傳送到外部服務",
        "gcp_solution": "Cloud DLP 掃描 + Tokenization Pipeline",
        "impact": "進入 RAG Pipeline 前必須先掃描、遮罩或 Tokenize PII"
    },
    "審計合規（Audit & Compliance）": {
        "constraint": "每個 AI 模型調用都要有審計日誌",
        "gcp_solution": "Cloud Audit Logs → BigQuery + SIEM 整合",
        "impact": "Vertex AI Data Access log 必須啟用，保留 7 年"
    },
    "資料主權（Data Residency）": {
        "constraint": "資料不能離開台灣/亞洲",
        "gcp_solution": "GCP Region 鎖定（asia-east1）+ VPC-SC Data Residency Policy",
        "impact": "所有服務必須在 asia-east1 部署"
    }
}

print("=" * 60)
print("銀行客戶的限制分類")
print("=" * 60)

for constraint_type, details in BANK_CONSTRAINTS.items():
    print(f"\n🔒 {constraint_type}")
    print(f"   客戶需求: {details['constraint']}")
    print(f"   GCP 解法: {details['gcp_solution']}")
    print(f"   架構影響: {details['impact']}")

## Part 2：PII 偵測與遮罩（Python 模擬 + Cloud DLP 參考）

In [ ]:
# ✅ PII 偵測器（Python 模擬 Cloud DLP）
# 生產環境使用 Google Cloud DLP API

@dataclass
class PIIMatch:
    pii_type: str
    original: str
    start: int
    end: int

class PIIScanner:
    """模擬 Cloud DLP 的 PII 偵測功能"""
    
    # 模擬偵測規則（Cloud DLP 有更完整的 ML-based 偵測）
    PATTERNS = {
        "PERSON_NAME": r'(?:甲方|乙方|簽約人|立約人)[：:]?\s*([\u4e00-\u9fff]{2,4})',
        "ID_NUMBER": r'[A-Z][12][0-9]{8}',  # 台灣身分證
        "PHONE_NUMBER": r'(?:0[789][0-9]{8}|\+886[0-9]{9})',
        "EMAIL": r'[\w.+-]+@[\w-]+\.[a-z]{2,}',
        "BANK_ACCOUNT": r'[0-9]{10,16}',
        "FINANCIAL_AMOUNT": r'新台幣[\s\u4e00-\u9fff]*元整',
    }
    
    def scan(self, text: str) -> list[PIIMatch]:
        """掃描文字中的 PII（模擬 DLP inspect）"""
        matches = []
        for pii_type, pattern in self.PATTERNS.items():
            for m in re.finditer(pattern, text):
                matches.append(PIIMatch(
                    pii_type=pii_type,
                    original=m.group(),
                    start=m.start(),
                    end=m.end()
                ))
        return matches
    
    def tokenize(self, text: str) -> tuple[str, dict]:
        """
        Tokenization：將 PII 替換為 token，保存還原表
        進 LLM 前替換，LLM 輸出後還原
        """
        token_map = {}  # token → 原始值
        reverse_map = {}  # 原始值 → token
        result = text
        
        matches = self.scan(text)
        # 從後往前替換（避免位置偏移）
        for match in sorted(matches, key=lambda m: m.start, reverse=True):
            original = match.original
            if original in reverse_map:
                token = reverse_map[original]
            else:
                token = f"[{match.pii_type}_{len(token_map):03d}]"
                token_map[token] = original
                reverse_map[original] = token
            result = result[:match.start] + token + result[match.end:]
        
        return result, token_map
    
    def restore(self, tokenized_text: str, token_map: dict) -> str:
        """將 LLM 輸出中的 token 還原為原始值"""
        result = tokenized_text
        for token, original in token_map.items():
            result = result.replace(token, original)
        return result


# 測試 PII 掃描
scanner = PIIScanner()

CONTRACT_WITH_PII = """合約正本
立約人：甲方 王大明（身分證字號：A123456789，電話：0912345678）
電子郵件：wang@example.com
銀行帳號：1234567890123

第 3 條 — 違約責任
甲方若未按時交付，乙方得向甲方請求違約金，
金額為新台幣五百萬元整。
"""

print("=" * 50)
print("PII 偵測測試")
print("=" * 50)

# 掃描 PII
pii_matches = scanner.scan(CONTRACT_WITH_PII)
print(f"\n找到 {len(pii_matches)} 個 PII：")
for match in pii_matches:
    print(f"  [{match.pii_type}] '{match.original}'")

# Tokenize
print("\nTokenization 後：")
tokenized, token_map = scanner.tokenize(CONTRACT_WITH_PII)
print(tokenized)

print("\nToken 還原表（不進入 LLM）：")
for token, original in token_map.items():
    print(f"  {token} → {original}")

In [ ]:
# 📋 Cloud DLP 真實 API 參考（不需要執行，只是參考）

print("""
=== Cloud DLP 真實 API 參考 ===

# Python SDK
from google.cloud import dlp_v2

dlp = dlp_v2.DlpServiceClient()

info_types = [
    {"name": "PERSON_NAME"},
    {"name": "PHONE_NUMBER"},
    {"name": "EMAIL_ADDRESS"},
    {"name": "FINANCIAL_ACCOUNT_NUMBER"},
    {"name": "TAIWAN_ID_NUMBER"},  # 台灣身分證
]

# 只掃描（不遮罩）
inspect_config = dlp_v2.InspectConfig(info_types=info_types)
response = dlp.inspect_content(
    request={
        "parent": f"projects/{PROJECT_ID}/locations/asia-east1",
        "inspect_config": inspect_config,
        "item": {"value": document_text}
    }
)

# 遮罩（替換為 ***）
deidentify_config = dlp_v2.DeidentifyConfig(
    info_type_transformations=dlp_v2.InfoTypeTransformations(
        transformations=[{
            "primitive_transformation": {
                "replace_with_info_type_config": {}  # 替換為 [PERSON_NAME] 等
            }
        }]
    )
)

關鍵：DLP 在文件 Indexing 時掃描（不是 Query 時），
結果快取，避免每次查詢都呼叫 DLP API（成本控制）
""")

## Part 3：Audit Log 設計

In [ ]:
# ✅ Audit Log 實作（模擬 Cloud Audit Logs）

@dataclass
class AuditLogEntry:
    """模擬 Cloud Audit Log 的格式"""
    log_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    user_id: str = ""              # 誰呼叫的
    service: str = ""              # 哪個服務
    method: str = ""               # 呼叫的方法
    resource: str = ""             # 被存取的資源
    request_hash: str = ""         # 請求的 hash（不存明文，保護隱私）
    response_status: str = ""      # 成功 / 失敗
    latency_ms: int = 0
    model_id: str = ""             # 使用了哪個 AI 模型
    tokens_used: int = 0
    pii_detected: bool = False     # 是否有偵測到 PII
    session_id: str = ""           # 用於追蹤同一個對話

# 模擬 Audit Log 存儲（生產中寫到 Cloud Logging）
audit_log_store: list[AuditLogEntry] = []

def write_audit_log(entry: AuditLogEntry):
    """模擬寫入 Cloud Logging（生產中透過 Python logging client）"""
    audit_log_store.append(entry)
    print(f"  📝 [Audit] {entry.timestamp[:19]} | {entry.user_id} | {entry.method} | "
          f"status={entry.response_status} | latency={entry.latency_ms}ms | "
          f"tokens={entry.tokens_used} | pii={entry.pii_detected}")

def query_audit_logs(user_id: str = None, date_from: str = None) -> list[AuditLogEntry]:
    """模擬合規查詢（生產中透過 BigQuery 查詢）"""
    results = audit_log_store
    if user_id:
        results = [e for e in results if e.user_id == user_id]
    return results

# 測試 Audit Log
print("Audit Log 測試：")

test_entry = AuditLogEntry(
    user_id="lawyer_001",
    service="aiplatform.googleapis.com",
    method="predict",
    resource="projects/bank-ai/locations/asia-east1/models/gemini-1.5-pro",
    request_hash=hashlib.sha256(b"contract content").hexdigest()[:16],
    response_status="OK",
    latency_ms=1250,
    model_id="gemini-1.5-pro",
    tokens_used=3200,
    pii_detected=True,
    session_id="session_abc123"
)

write_audit_log(test_entry)

print("\n合規查詢：lawyer_001 在過去 7 天的 AI 使用記錄")
logs = query_audit_logs(user_id="lawyer_001")
print(f"找到 {len(logs)} 筆記錄")

In [ ]:
# 📋 GCP Audit Log 設定指令（參考）

print("""
=== GCP Audit Log 設定 ===

# 啟用 Vertex AI 的 Data Access Audit Log
# policy.yaml：
auditConfigs:
- service: aiplatform.googleapis.com
  auditLogConfigs:
  - logType: DATA_READ   # 記錄模型調用的 input
  - logType: DATA_WRITE  # 記錄模型調用的 output

# 套用政策
gcloud projects set-iam-policy ${PROJECT_ID} policy.yaml

# 建立 Log Sink → BigQuery（供合規查詢）
gcloud logging sinks create vertex-ai-audit-sink \\
  bigquery.googleapis.com/projects/${PROJECT_ID}/datasets/audit_logs \\
  --log-filter='resource.type="aiplatform.googleapis.com"' \\
  --project=${PROJECT_ID}

# 建立 Log Sink → Pub/Sub（供 SIEM 即時監控）
gcloud logging sinks create vertex-ai-siem-sink \\
  pubsub.googleapis.com/projects/${PROJECT_ID}/topics/siem-events \\
  --log-filter='resource.type="aiplatform.googleapis.com"' \\
  --project=${PROJECT_ID}

合規團隊查詢範例（BigQuery SQL）：
SELECT
  timestamp,
  proto_payload.auth_info.principal_email AS user,
  proto_payload.method_name AS method,
  proto_payload.resource_name AS resource
FROM `project.audit_logs.cloudaudit_googleapis_com_data_access`
WHERE DATE(timestamp) >= '2026-01-01'
  AND service_name = 'aiplatform.googleapis.com'
ORDER BY timestamp DESC
LIMIT 100
""")

## Part 4：VPC Service Controls — 資料不出 VPC 的核心技術

In [ ]:
print("""
=== VPC Service Controls (VPC-SC) 設計 ===

核心概念：
  VPC-SC 建立一個「安全邊界」（Security Perimeter）
  邊界內的 GCP 服務只能被邊界內的資源存取
  即使 IAM 憑證外洩，邊界外的人也無法存取資料

沒有 VPC-SC：
  App（在 VPC 內）→ Vertex AI API（走公共網路）
  ❌ 資料經過公共網路
  ❌ 銀行 IT 安全政策不允許

有 VPC-SC：
  ┌──────────────────────────────────────┐
  │  VPC Service Perimeter               │
  │  ┌──────────────┐  ┌─────────────┐  │
  │  │  Customer VPC │  │ Vertex AI   │  │
  │  │  (App / GKE)  │→ │ Gemini API  │  │
  │  │               │  │ Embedding   │  │
  │  │               │  │ Vector Search│ │
  │  └──────────────┘  └─────────────┘  │
  │  邊界外無法存取邊界內的任何資料        │
  └──────────────────────────────────────┘
  ✅ 資料不離開 GCP Organization
  ✅ 符合銀行安全政策

=== GCP CLI 設定指令 ===

# 1. 建立 Access Policy
gcloud access-context-manager policies create \\
  --organization=${ORG_ID} \\
  --title="Bank AI Policy"

# 2. 建立 Service Perimeter
gcloud access-context-manager perimeters create bank-ai-perimeter \\
  --policy=${POLICY_NAME} \\
  --title="Bank AI Perimeter" \\
  --resources=projects/${PROJECT_NUMBER} \\
  --restricted-services=aiplatform.googleapis.com,storage.googleapis.com \\
  --access-levels=${ACCESS_LEVEL}

# 3. 確認 VPC 流量只走 restricted.googleapis.com
# 在 VPC 的 DNS 設定：
# *.googleapis.com → 199.36.153.4/30 (restricted.googleapis.com)
"""
)

## Part 5：Private Service Connect — 讓流量完全不出 VPC

In [ ]:
print("""
=== Private Service Connect (PSC) 設計 ===

為什麼需要 PSC（VPC-SC 不夠嗎）？
  VPC-SC: 控制「誰能存取服務」（Access Control）
  PSC:    控制「流量走哪條路」（Network Path）

  沒有 PSC：
  Your VPC → (通過公共網路) → Vertex AI Endpoint
  即使有 VPC-SC，流量路徑仍然可能經過公共網路

  有 PSC：
  Your VPC → PSC Endpoint (Private IP 10.0.0.100) → Vertex AI
  整個路徑在 Google 骨幹網路內，完全不碰公網

=== GCP CLI 設定指令 ===

# 1. 在 VPC 內建立 PSC Endpoint（分配私有 IP）
gcloud compute addresses create vertex-ai-psc-ip \\
  --region=asia-east1 \\
  --subnet=my-ai-subnet \\
  --addresses=10.0.1.100

# 2. 建立 Forwarding Rule 指向 Vertex AI
gcloud compute forwarding-rules create vertex-ai-psc-rule \\
  --region=asia-east1 \\
  --network=my-vpc \\
  --address=vertex-ai-psc-ip \\
  --target-google-apis-bundle=all-apis

# 3. 應用程式改用 PSC 私有 IP
import vertexai
vertexai.init(
    project="bank-ai-project",
    location="asia-east1",
    api_endpoint="10.0.1.100"  # PSC 私有 IP，不走公網
)

# 4. 設定 DNS 讓 *.googleapis.com 解析到 PSC 私有 IP
gcloud dns managed-zones create private-googleapis \\
  --dns-name=googleapis.com. \\
  --visibility=private \\
  --networks=my-vpc \\
  --description="Route googleapis.com to PSC"
"""
)

## Part 6：完整 RAG Pipeline（VPC 限制版模擬）

In [ ]:
# ✅ VPC 限制版 RAG Pipeline 完整流程模擬

class BankRAGPipeline:
    """
    符合銀行安全政策的 RAG Pipeline
    三個限制都有對應的技術措施：
    1. 網路隔離 → 模擬 PSC 私有端點
    2. PII 保護 → Tokenization
    3. 審計日誌 → 每次呼叫都記錄
    """
    
    def __init__(self):
        self.scanner = PIIScanner()
        self.use_private_endpoint = True  # 模擬 PSC
        self.vector_store = {}  # 模擬 Vertex AI Vector Search
    
    def _simulate_vertex_ai_call(self, prompt: str, session_id: str, user_id: str) -> str:
        """模擬透過 PSC 私有端點呼叫 Vertex AI Gemini"""
        endpoint = "10.0.1.100" if self.use_private_endpoint else "aiplatform.googleapis.com"
        
        start = time.time()
        
        # 模擬 LLM 回應（生產中用 vertexai.GenerativeModel）
        response = f"[模擬 Vertex AI 回應] 根據合約資料分析：{prompt[:50]}..."
        latency = int((time.time() - start) * 1000) + 800  # 模擬 800ms
        
        # 寫入 Audit Log（每次呼叫都記錄）
        write_audit_log(AuditLogEntry(
            user_id=user_id,
            service="aiplatform.googleapis.com",
            method="predict",
            resource=f"projects/bank-ai/locations/asia-east1/models/gemini-1.5-pro",
            request_hash=hashlib.sha256(prompt.encode()).hexdigest()[:16],
            response_status="OK",
            latency_ms=latency,
            model_id="gemini-1.5-pro",
            tokens_used=len(prompt.split()) * 2,
            pii_detected=False,  # 已在前面 Tokenize
            session_id=session_id
        ))
        
        return response
    
    def index_document(self, document_id: str, content: str, user_id: str) -> dict:
        """
        文件 Indexing Pipeline：
        1. PII 掃描
        2. Tokenization
        3. Embedding（模擬）
        4. 存入 Vector Store
        """
        print(f"[Indexing] 文件 {document_id}")
        
        # Step 1: PII 掃描
        matches = self.scanner.scan(content)
        print(f"  PII 偵測: {len(matches)} 個")
        
        # Step 2: Tokenize PII
        tokenized_content, token_map = self.scanner.tokenize(content)
        print(f"  Tokenize 完成: {len(token_map)} 個 token")
        
        # Step 3: 生成 Embedding（模擬 Vertex AI Embedding）
        # 生產中：vertexai.TextEmbeddingModel.from_pretrained('text-embedding-004')
        embedding = [0.1] * 768  # 模擬 768 維向量
        
        # Step 4: 存入 Vector Store（模擬 Vertex AI Vector Search）
        self.vector_store[document_id] = {
            "tokenized_content": tokenized_content,
            "token_map": token_map,
            "embedding": embedding,
            "original_size": len(content),
        }
        
        print(f"  ✅ Indexing 完成")
        return {"document_id": document_id, "pii_found": len(matches)}
    
    def query(self, question: str, user_id: str, session_id: str) -> str:
        """
        查詢 Pipeline：
        1. 檢索相關文件（模擬 Vector Search）
        2. 構建 Prompt（Tokenized Context）
        3. 呼叫 LLM（透過 PSC 私有端點）
        4. 還原 Token
        """
        print(f"\n[Query] 用戶={user_id}, 問題='{question}'")
        
        # Step 1: 檢索（模擬 Vector Search）
        if not self.vector_store:
            return "知識庫為空"
        doc_id = list(self.vector_store.keys())[0]  # 模擬檢索
        doc = self.vector_store[doc_id]
        print(f"  檢索到文件: {doc_id}")
        
        # Step 2: 構建 Prompt（用 Tokenized 內容，不含真實 PII）
        prompt = f"""根據以下合約資料回答問題。
合約資料（PII 已 Tokenize）：
{doc['tokenized_content']}

問題：{question}"""
        
        # Step 3: 呼叫 LLM（模擬 PSC 私有端點，並記錄 Audit Log）
        print(f"  透過 PSC 私有端點呼叫 Vertex AI...")
        response = self._simulate_vertex_ai_call(prompt, session_id, user_id)
        
        # Step 4: 如果需要，還原回應中的 Token
        restored_response = self.scanner.restore(response, doc["token_map"])
        
        return restored_response


# 執行完整流程
print("=" * 55)
print("VPC 限制版 RAG Pipeline 完整演示")
print("=" * 55)

pipeline = BankRAGPipeline()

# 1. 索引合約文件
pipeline.index_document("contract_001", CONTRACT_WITH_PII, user_id="system")

# 2. 用戶查詢
answer = pipeline.query(
    question="這份合約的違約金是多少？",
    user_id="lawyer_001",
    session_id="session_abc123"
)
print(f"\n回答: {answer}")

# 3. 審計查詢
print(f"\n審計記錄數: {len(audit_log_store)} 筆")

## Part 7：面試白板語言 + 速查表

In [ ]:
print("""
面試答題框架（Constraint-First 四步法）：
─────────────────────────────────────────────────────
第一步：分類限制
  銀行給的限制有三類：
  ├── 網路隔離（API 不走公網）
  ├── 資料分類（PII 不進外部服務）
  └── 審計合規（每次 AI 呼叫都要記錄）

第二步：確認每個限制的核心技術
  網路隔離 → VPC Service Controls + Private Service Connect
  PII 保護 → Cloud DLP 掃描 + Tokenization Pipeline
  審計日誌 → Cloud Audit Logs → BigQuery + SIEM 整合

第三步：畫出資料流
  從用戶瀏覽器到 LLM，再回來的完整路徑：
  用戶 → Internal LB → Cloud Run（VPC Connector）
       → Cloud DLP 掃描 → Tokenize PII
       → PSC 私有端點 → Vertex AI Gemini（在 VPC-SC 邊界內）
       → Audit Log 記錄
       → 回傳給用戶（還原 Token）

第四步：和客戶確認
  「這個架構是否符合你們的安全政策？」
  FDE 提供技術方案，合規確認由客戶的法務和安全團隊決定。

重要：Vertex AI 在 VPC-SC Perimeter 內 ≠ 外部服務
  ├── 資料不離開 GCP Organization
  ├── Google 不用客戶資料訓練模型（可簽 BAA）
  └── 對銀行 IT 的說法：「資料在你們的 GCP 組織邊界內」
─────────────────────────────────────────────────────
""")

print("銀行/金融客戶限制速查表：")
quickref = [
    ("資料不能出台灣",         "asia-east1 + VPC-SC Data Residency Policy"),
    ("API 不能走公網",          "Private Service Connect + VPC-SC"),
    ("PII 不能給外部 AI",       "Cloud DLP + Tokenization + VPC-SC"),
    ("每次 AI 調用要有 Log",    "Cloud Audit Logs → BigQuery + SIEM"),
    ("必須用自己的模型",         "Vertex AI Custom Model or GKE 自建推論"),
    ("所有通訊必須加密",         "TLS 1.2+ + CMEK（靜態加密）"),
    ("不能用任何第三方服務",     "All-GCP Stack（Vertex AI + BigQuery + GCS）"),
]

print(f"\n{'限制':<20} {'GCP 解法'}")
print("-" * 60)
for constraint, solution in quickref:
    print(f"  {constraint:<20} {solution}")